<a href="https://colab.research.google.com/github/GustavoABrandao/Projeto---IA-Facens/blob/main/Projeto%20Big%20Data.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [35]:
# Importando bibliotecas necessárias
from pyspark.sql import SparkSession
from pyspark.ml.feature import VectorAssembler
from pyspark.ml.classification import RandomForestClassifier
from pyspark.ml.evaluation import MulticlassClassificationEvaluator
from pyspark.ml.feature import StringIndexer, OneHotEncoder, VectorAssembler
from pyspark.ml import Pipeline
from pyspark.sql.functions import expr
from pyspark.sql.functions import col, regexp_extract
from pyspark.sql.functions import expr, current_date, year as spark_year
import matplotlib.pyplot as plt
from pyspark.sql.functions import udf
from pyspark.sql.types import IntegerType

spark = SparkSession.builder \
    .appName("PySpark Docker Example") \
    .getOrCreate()

ConnectionRefusedError: [Errno 111] Connection refused

In [ ]:
dataframe = spark.read.csv("/content/vehicles.csv", header=True, inferSchema=True)
num_linhas = dataframe.count()
print(f"Número de linhas no DataFrame: {num_linhas}")

In [ ]:
dataframe.show(50)

In [ ]:
df = dataframe

# Regex mais flexível: ignora o que não for número/ponto/sinal de menos
# Removemos o sinal de negativo de colunas que não podem ser negativas (preço, ano, km)
df = df.withColumn(
    "year",
    expr("try_cast(regexp_replace(year, '[^0-9]', '') as double)")
)

df = df.withColumn(
    "price",
    expr("try_cast(regexp_replace(price, '[^0-9.]', '') as double)")
)

df = df.withColumn(
    "odometer",
    expr("try_cast(regexp_replace(odometer, '[^0-9.]', '') as double)")
)

df = df.withColumn(
    "lat",
    expr("try_cast(regexp_replace(lat, '[^0-9.-]', '') as double)")
)

df = df.withColumn(
    "long",
    expr("try_cast(regexp_replace(long, '[^0-9.-]', '') as double)")
)

df = df.withColumn(
    "car_age",
    spark_year(current_date()) - col("year")
)
# Remoção de colunas
df = df.drop(
    "id", "url", "region_url", "VIN",
    "image_url", "description", "county",
    "posting_date", "size"
)

In [ ]:
# =========================
# 2. DROP E FILTROS (MENOS AGRESSIVO)
# =========================

# DICA: Removi lat e long do dropna. Se não tiver localização, a gente mantém o carro!
df = df.dropna(subset=["price", "year", "odometer"])

# Filtros de sanidade
df = df.filter(
    (col("price") >= 100.0) & (col("price") < 1000000.0) &
    (col("year") > 2005.0) & (col("year") <= 2026.0)
)


# VERIFICAÇÃO INTERMEDIÁRIA
contagem = df.count()
print(f"Registros após limpeza inicial: {contagem}")

if contagem > 0:
    # 3. CACHE E ESTATÍSTICA
    df.cache()

    for c in ["price", "odometer"]:
        # Verificamos se há dados suficientes para calcular quantis
        quantis = df.approxQuantile(c, [0.10, 0.90], 0.05)
        if len(quantis) == 2:
            q_low, q_high = quantis[0], quantis[1]
            df = df.filter((col(c) >= q_low) & (col(c) <= q_high))
            print(f"Filtro aplicado em {c}: {q_low} até {q_high}")

    print(f"Registros finais: {df.count()}")
else:
    print("ERRO: O filtro deletou todos os dados! Verifique as colunas de entrada.")
    df.show(5) # Mostra o que restou para você entender o problema

In [ ]:
# =========================
# 3. Colunas categóricas
# =========================

# One-hot (nominais)
categorical_cols = [
    "region", "manufacturer", "fuel", "transmission",
    "drive", "type", "paint_color", "state", "title_status"
]

# StringIndexer (ordinal ou alta cardinalidade)
index_cols = ["condition", "model"]




In [ ]:
# =========================
# 4. Criar stages
# =========================
stages = []

# Index + OneHot para categóricas nominais
for col_name in categorical_cols:
    indexer = StringIndexer(
        inputCol=col_name,
        outputCol=col_name + "_index",
        handleInvalid="keep"
    )

    encoder = OneHotEncoder(
        inputCol=col_name + "_index",
        outputCol=col_name + "_onehot",
        dropLast=True
    )

    stages += [indexer, encoder]

# Apenas index para ordinal/alta cardinalidade
for col_name in index_cols:
    indexer = StringIndexer(
        inputCol=col_name,
        outputCol=col_name + "_index",
        handleInvalid="keep"
    )

    stages.append(indexer)



In [ ]:
# =========================
# 5. Pipeline
# =========================
pipeline = Pipeline(stages=stages)
model = pipeline.fit(df)
df_prepared = model.transform(df)
df_prepared.createOrReplaceTempView("cars_prepared")


In [ ]:
features_sql = spark.sql("""
SELECT *
FROM cars_prepared
LIMIT 20
""")

features_sql.show()


In [ ]:
df_prepared.createOrReplaceTempView("view_treino")

# Selecionamos as versões para o modelo
df_pre_treino = spark.sql("""
    SELECT
        price,
        car_age,
        odometer,
        lat,
        long,
        region_onehot,
        manufacturer_onehot,
        fuel_onehot,
        transmission_index,
        drive_onehot,
        type_onehot,
        paint_color_onehot,
        state_onehot,
        title_status_onehot,
        condition_index,
        model_index
    FROM view_treino
""")

df_pre_treino.show(50)


In [ ]:
from pyspark.sql.functions import col, udf
from pyspark.sql.types import DoubleType

# Garante que a UDF para ler os vetores OneHot esteja definida
get_index = udf(lambda v: float(v.indices[0]) if len(v.indices) > 0 else -1.0, DoubleType())

# Combinação de todos os filtros em uma única operação
df_pre_treino = df_pre_treino \
    .withColumn("paint_temp_idx", get_index(col("paint_color_onehot"))) \
    .withColumn("drive_temp_idx", get_index(col("drive_onehot"))) \
    .filter(
        (~col("paint_temp_idx").isin([5.0, 6.0, 7.0, 8.0, 9.0, 10.0, 11.0])) &
        (~col("drive_temp_idx").isin([2.0])) &
        (~col("condition_index").isin([2.0, 3.0, 4.0, 5.0]))
    ) \
    .drop("paint_temp_idx", "drive_temp_idx")

# Opcional: Verifique o resultado
# print(f"Linhas após filtros: {df_pre_treino.count()}")

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

data_norm = []

cols = ["price", "odometer", "car_age"]

for c in cols:
    sample = df_pre_treino.select(c) \
        .dropna() \
        .sample(False, 0.05, seed=42) \
        .rdd.map(lambda x: x[0]) \
        .collect()

    sample = np.array(sample)

    std = sample.std()

    if std != 0:
        sample = (sample - sample.mean()) / std
    else:
        sample = sample - sample.mean()  # evita divisão por zero

    data_norm.append(sample)

plt.figure(figsize=(8,5))
plt.boxplot(data_norm)
plt.xticks(range(1, len(cols)+1), cols)
plt.title("Boxplot normalizado")
plt.ylabel("Valores normalizados")
plt.show()

In [36]:

# Converte vetor one-hot para índice da posição ativa
def onehot_to_index(v):
    if v is None:
        return None
    arr = v.toArray()
    if arr.sum() == 0:
        return -1
    return int(arr.argmax())

onehot_to_index_udf = udf(onehot_to_index, IntegerType())

onehot_cols = [
    "region_onehot",
    "manufacturer_onehot",
    "fuel_onehot",
    "drive_onehot",
    "type_onehot",
    "paint_color_onehot",
    "state_onehot",
    "title_status_onehot"
]

index_cols = [
    "transmission_index",
    "condition_index",
    "model_index"
]

df_plot = df_pre_treino

for c in onehot_cols:
    df_plot = df_plot.withColumn(c + "_class", onehot_to_index_udf(col(c)))

plot_cols = [c + "_class" for c in onehot_cols] + index_cols

for c in plot_cols:
    data = df_plot.groupBy(c) \
        .count() \
        .orderBy("count", ascending=False) \
        .collect()

    x = [str(row[c]) for row in data]
    y = [row["count"] for row in data]

    plt.figure(figsize=(10, 5))
    plt.bar(x, y)
    plt.title(f"Distribuição de {c}")
    plt.xlabel("Classe")
    plt.ylabel("Quantidade")
    plt.xticks(rotation=45)
    plt.show()

ConnectionRefusedError: [Errno 111] Connection refused

In [ ]:
# Lista de colunas que não podem ser nulas para o treino
cols_to_check = [
    "car_age", "odometer", "lat", "long",
    "manufacturer_onehot", "drive_onehot",
    "paint_color_onehot", "condition_index", "price"
]

# Remove as linhas que possuem null em qualquer uma dessas colunas
df_pre_treino_clean = df_pre_treino.dropna(subset=cols_to_check)

# Agora use o df_pre_treino_clean no seu transform
df_model = assembler.transform(df_pre_treino_clean).select("features", "price")

In [ ]:
df_pre_treino.count()


Treinamento de Random Forest

In [ ]:
from pyspark.ml.feature import VectorAssembler
from pyspark.ml.regression import RandomForestRegressor
from pyspark.ml.evaluation import RegressionEvaluator

# Lista dos atributos que você definiu (Features)
feature_cols = [
    "car_age",
    "odometer",
    "lat",
    "long",
    "manufacturer_onehot",
    "drive_onehot",
    "paint_color_onehot",
    "condition_index"
]

# Assembler: agrupa tudo em uma coluna chamada 'features'
assembler = VectorAssembler(inputCols=feature_cols, outputCol="features")

# Transforma o DataFrame filtrado
df_model = assembler.transform(df_pre_treino).select("features", "price")

In [ ]:
# 80% para treino e 20% para teste
train_data, test_data = df_model.randomSplit([0.8, 0.2], seed=42)

In [ ]:
from pyspark import StorageLevel
from pyspark.ml.regression import RandomForestRegressor

# 1. Limpa caches antigos, se ainda existir conexão com Spark
try:
    df_pre_treino.unpersist()
except:
    pass

# 2. Reparticiona os dados de treino
train_data_repartitioned = train_data.repartition(8)

# 3. Configuração do modelo
rf = RandomForestRegressor(
    featuresCol="features",
    labelCol="price",
    numTrees=120,
    maxDepth=13,
    seed=42
)

# 4. Treino
model = rf.fit(train_data_repartitioned)

In [ ]:
# Previsões no conjunto de teste
predictions = model.transform(test_data)

# Avaliadores
evaluator_rmse = RegressionEvaluator(labelCol="price", predictionCol="prediction", metricName="rmse")
evaluator_r2 = RegressionEvaluator(labelCol="price", predictionCol="prediction", metricName="r2")

rmse = evaluator_rmse.evaluate(predictions)
r2 = evaluator_r2.evaluate(predictions)

print(f"RMSE (Erro médio): {rmse:.2f}")
print(f"R² (Precisão do modelo): {r2:.2f}")

# Mostrar as primeiras comparações entre Real e Previsto
predictions.select("price", "prediction").show(10)